##### Ingest qualifying json files

##### Read the JSON file using the spark dataframe reader API

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-28")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
qualifying_schema = StructType(fields=[StructField("qualifyId", IntegerType(), False),
                                      StructField("raceId", IntegerType(), True),
                                      StructField("driverId", IntegerType(), True),
                                      StructField("constructorId", IntegerType(), True),
                                      StructField("number", IntegerType(), True),
                                      StructField("position", IntegerType(), True),
                                      StructField("q1", StringType(), True),
                                      StructField("q2", StringType(), True),
                                      StructField("q3", StringType(), True),
                                     ])

In [0]:
qualifying_df = spark.read \
.schema(qualifying_schema) \
.option("multiLine", True) \
.json(f"{raw_path}/qualifying")

##### Rename columns and add new columns

In [0]:
final_df = qualifying_df.withColumnRenamed("qualifyId", "qualify_id") \
.withColumnRenamed("driverId", "driver_id") \
.withColumnRenamed("raceId", "race_id") \
.withColumnRenamed("constructorId", "constructor_id") \
.withColumn("ingestion_date", current_timestamp()) \
.withColumn("data_source", lit(v_data_source))

In [0]:
final_df = add_ingestion_date(final_df)

##### Write to output to processed container 

In [0]:
# final_df.write.mode("overwrite").parquet(f"{processed_path}/qualifying")

In [0]:
### final solution 
overwrite_partition_free(final_df, 'f1_processed', 'qualifying', 'qualify_id')

##### Check

In [0]:
# spark.read.parquet(f"{processed_path}/qualifying").display()

In [0]:
dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
select race_id, count(1) as count
from f1_processed.qualify_id
group by race_id
order by race_id desc;